In [1]:
!git clone https://github.com/VladWero08/time-series-ad-gan.git

fatal: destination path 'time-series-ad-gan' already exists and is not an empty directory.


In [2]:
pip install pandas numpy kagglehub torch scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys 
import os

module_path = os.path.abspath("./time-series-ad-gan")
if module_path not in sys.path:
    sys.path.append(module_path)

In [5]:
import ast
import pandas as pd
import numpy as np
import json
import kagglehub
import typing as t
import torch
import matplotlib.pyplot as plt
from urllib.request import urlopen

from src.models.tad_gan import run_pipeline
from src.utils.data import intervals_to_points
from src.utils.errors import point_wise_error, area_wise_error, dtw_error

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Working on {device}!")

Working on cuda!


In [7]:
def plot_signal(X: np.ndarray, y: np.ndarray) -> None:
    plt.figure(figsize=(15, 4))
    for idx in np.where(y == 1)[0]:
        plt.axvline(idx, color='red', alpha=0.4, linewidth=0.8, zorder=0)
    plt.plot(X, zorder=1)
    plt.show()

## **Yahoo S5**

In [8]:
A1_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A1Benchmark/"
A1_FILE_NAME = "real_"
A1_N_FILES = 67

A2_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A2Benchmark/"
A2_FILE_NAME = "synthetic_"
A2_N_FILES = 100

A3_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A3Benchmark/"
A3_FILE_NAME = "A3Benchmark-TS"
A3_N_FILES = 100

A4_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A4Benchmark/"
A4_FILE_NAME = "A4Benchmark-TS"
A4_N_FILES = 100

In [9]:
YAHOO_HYPERPARAMS = {
    "train_ratio": 0.6,
    "device": device,
    "anomaly_type": "contextual",
    "verbose": False,
    "lr_gg": 1e-5,
    "lr_gf": 1e-5,
    "lr_dx": 1e-5,
    "lr_dz": 1e-5,
    "batch_size": 512,
    "epochs": 1000,
    "rec_error_funcs": [("point", point_wise_error), ("area", area_wise_error)],
}

In [10]:
def yahoo_download_subdataset(folder: str, file_name: str, n_files: int) -> t.List[pd.DataFrame]:
    yahoo_signals = []
    yahoo_points = 0
    yahoo_anomalies = 0

    for i in range(1, n_files + 1):
        # name of the .csv in the repository
        yahoo_fn = f"{file_name}{i}.csv"
        # url to the raw .csv file in the repository
        yahoo_url = f"{folder}{yahoo_fn}"
        yahoo_df = pd.read_csv(yahoo_url)
        yahoo_df = yahoo_df.rename(columns={"anomaly": "is_anomaly", "timestamps": "timestamp"})
        
        # count the number of points and the number of anomaly points
        yahoo_points += len(yahoo_df)
        yahoo_anomalies += (yahoo_df["is_anomaly"] == 1).sum()
        yahoo_signals.append(yahoo_df)

        if i % 10 == 0:
            print(f"Downloaded {i} .csv files.")

    print()
    print("Finished downloading!")
    print("---------------------")
    print(f"A1 Total Signals: {A1_N_FILES}")
    print(F"A1 Total Points: {yahoo_points}")
    print(f"A1 Total Anomaly Points: {yahoo_anomalies}")
    print(f"A1 Anomaly Rate: {(yahoo_anomalies / yahoo_points) * 100:.2f}%")

    return yahoo_signals

### **A1**

In [11]:
a1_signals = yahoo_download_subdataset(folder=A1_FOLDER, file_name=A1_FILE_NAME, n_files=A1_N_FILES)

Downloaded 10 .csv files.
Downloaded 20 .csv files.
Downloaded 30 .csv files.
Downloaded 40 .csv files.
Downloaded 50 .csv files.
Downloaded 60 .csv files.

Finished downloading!
---------------------
A1 Total Signals: 67
A1 Total Points: 94866
A1 Total Anomaly Points: 1669
A1 Anomaly Rate: 1.76%


In [12]:
total_metrics = {name: np.zeros(3) for (name, _) in YAHOO_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a1_signal in enumerate(a1_signals):
    X = a1_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a1_signal['is_anomaly'].to_numpy()

    print(f"A1 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **YAHOO_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A1 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

A1 Signal 1...


/venv/main/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


A1 Signal 2...
A1 Signal 3...
A1 Signal 4...
A1 Signal 5...
A1 Signal 6...
A1 Signal 7...
A1 Signal 8...
A1 Signal 9...
A1 Signal 10...
A1 Signal 11...
A1 Signal 12...
A1 Signal 13...
A1 Signal 14...
A1 Signal 15...
A1 Signal 16...
A1 Signal 17...
A1 Signal 18...
A1 Signal 19...
A1 Signal 20...
A1 Signal 21...
A1 Signal 22...
A1 Signal 23...
A1 Signal 24...
A1 Signal 25...
A1 Signal 26...
A1 Signal 27...
A1 Signal 28...
A1 Signal 29...
A1 Signal 30...
A1 Signal 31...
A1 Signal 32...
A1 Signal 33...
A1 Signal 34...
A1 Signal 35...
A1 Signal 36...
A1 Signal 37...
A1 Signal 38...
A1 Signal 39...
A1 Signal 40...
A1 Signal 41...
A1 Signal 42...
A1 Signal 43...
A1 Signal 44...
A1 Signal 45...
A1 Signal 46...
A1 Signal 47...
A1 Signal 48...
A1 Signal 49...
A1 Signal 50...
A1 Signal 51...
A1 Signal 52...
A1 Signal 53...
A1 Signal 54...
A1 Signal 55...
A1 Signal 56...
A1 Signal 57...
A1 Signal 58...
A1 Signal 59...
A1 Signal 60...
A1 Signal 61...
A1 Signal 62...
A1 Signal 63...
A1 Signal 64...


### **A2**

In [ ]:
A2_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A2_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
a2_signals = yahoo_download_subdataset(folder=A2_FOLDER, file_name=A2_FILE_NAME, n_files=A2_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in A2_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a2_signal in enumerate(a2_signals):
    X = a2_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a2_signal['is_anomaly'].to_numpy()
    
    print(f"A2 Signal {i + 1}...")
        
    try:
        metrics = run_pipeline(X, y, **A2_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A2 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **A3**

In [ ]:
A3_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A3_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
a3_signals = yahoo_download_subdataset(folder=A3_FOLDER, file_name=A3_FILE_NAME, n_files=A3_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in A3_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a3_signal in enumerate(a3_signals):
    X = a3_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a3_signal['is_anomaly'].to_numpy()
    
    print(f"A3 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **A3_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A3 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **A4**

In [ ]:
A4_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A4_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
a4_signals = yahoo_download_subdataset(folder=A4_FOLDER, file_name=A4_FILE_NAME, n_files=A4_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in A4_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a4_signal in enumerate(a4_signals):
    X = a4_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a4_signal['is_anomaly'].to_numpy()
    
    print(f"A4 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **A4_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A4 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")